In [ ]:
```xml
<VSCode.Cell language="markdown">
# 💰 FinOps & Cost Optimization Advanced Guide

**Implementing cost-aware architecture with reserved instances, spot instances, and chargeback**

This guide covers:
- Cost allocation and tagging
- Reserved instance optimization
- Spot instance management
- Chargeback models
- Cost anomaly detection
- Budget forecasting
- Cost optimization recommendations
- Multi-cloud cost management
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Cost Allocation & Tagging

### Cost Tracking System
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List
from datetime import datetime
from enum import Enum

class CostType(Enum):
    COMPUTE = "compute"
    STORAGE = "storage"
    NETWORK = "network"
    DATABASE = "database"
    OTHER = "other"

@dataclass
class ResourceTag:
    """Resource tag for cost allocation"""
    key: str
    value: str

@dataclass
class ResourceCost:
    """Cost for a resource"""
    resource_id: str
    resource_type: str
    cost_type: CostType
    amount: float
    timestamp: datetime
    tags: List[ResourceTag] = field(default_factory=list)
    cost_center: Optional[str] = None
    project: Optional[str] = None

class CostAllocator:
    """Allocate costs to departments"""
    
    def __init__(self):
        self.costs: List[ResourceCost] = []
        self.tags: Dict[str, Dict] = {}
    
    def record_cost(self, cost: ResourceCost) -> bool:
        """Record resource cost"""
        
        self.costs.append(cost)
        return True
    
    def tag_resource(self, resource_id: str, tags: Dict[str, str]) -> bool:
        """Tag resource for cost allocation"""
        
        resource_tags = []
        for key, value in tags.items():
            resource_tags.append(ResourceTag(key, value))
        
        self.tags[resource_id] = {
            'tags': resource_tags,
            'tagged_at': datetime.utcnow().isoformat()
        }
        
        return True
    
    def get_cost_by_tag(self, tag_key: str, tag_value: str) -> float:
        """Get total cost by tag"""
        
        total = 0.0
        
        for cost in self.costs:
            for tag in cost.tags:
                if tag.key == tag_key and tag.value == tag_value:
                    total += cost.amount
        
        return total
    
    def get_cost_by_cost_center(self, cost_center: str) -> float:
        """Get cost by cost center"""
        
        return sum(c.amount for c in self.costs if c.cost_center == cost_center)
    
    def get_cost_by_project(self, project: str) -> float:
        """Get cost by project"""
        
        return sum(c.amount for c in self.costs if c.project == project)
    
    def get_cost_breakdown(self) -> Dict[str, float]:
        """Get cost breakdown by type"""
        
        breakdown = {}
        
        for cost in self.costs:
            cost_type = cost.cost_type.value
            breakdown[cost_type] = breakdown.get(cost_type, 0.0) + cost.amount
        
        return breakdown

class ChargebackModel:
    """Chargeback model for cost recovery"""
    
    def __init__(self):
        self.departments: Dict[str, Dict] = {}
        self.allocation_rules: Dict[str, callable] = {}
    
    def register_department(self, dept_id: str, name: str, budget: float):
        """Register department"""
        
        self.departments[dept_id] = {
            'name': name,
            'budget': budget,
            'charges': 0.0
        }
    
    def allocate_charge(self, dept_id: str, amount: float) -> bool:
        """Allocate charge to department"""
        
        if dept_id not in self.departments:
            return False
        
        self.departments[dept_id]['charges'] += amount
        
        return True
    
    def get_department_utilization(self, dept_id: str) -> float:
        """Get budget utilization"""
        
        if dept_id not in self.departments:
            return 0.0
        
        dept = self.departments[dept_id]
        
        if dept['budget'] == 0:
            return 100.0
        
        return (dept['charges'] / dept['budget']) * 100
    
    def get_over_budget_departments(self) -> List[str]:
        """Get departments over budget"""
        
        over_budget = []
        
        for dept_id, dept in self.departments.items():
            if dept['charges'] > dept['budget']:
                over_budget.append(dept_id)
        
        return over_budget
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Reserved & Spot Instances

### Instance Optimization
```python
class ReservedInstance:
    """Reserved instance"""
    
    def __init__(self, instance_type: str, count: int, term_years: int, 
                 hourly_rate: float):
        self.instance_type = instance_type
        self.count = count
        self.term_years = term_years
        self.hourly_rate = hourly_rate
        self.purchase_date = datetime.utcnow()
    
    def get_total_cost(self) -> float:
        """Calculate total cost"""
        hours_per_year = 365 * 24
        return self.hourly_rate * self.count * hours_per_year * self.term_years

class SpotInstance:
    """Spot instance"""
    
    def __init__(self, instance_type: str, count: int, max_price: float):
        self.instance_type = instance_type
        self.count = count
        self.max_price = max_price
        self.current_price = 0.0
        self.is_active = False

class InstanceOptimizer:
    """Optimize instance purchases"""
    
    def __init__(self):
        self.reserved_instances: List[ReservedInstance] = []
        self.spot_instances: List[SpotInstance] = []
        self.on_demand_costs: Dict[str, float] = {}
    
    def calculate_ri_recommendation(self, instance_type: str, 
                                   current_usage_hours: int,
                                   on_demand_rate: float) -> Dict:
        """Recommend reserved instance"""
        
        # Assume usage will continue
        annual_on_demand_cost = on_demand_rate * current_usage_hours
        
        # 1-year RI is ~30% cheaper
        ri_1year_rate = on_demand_rate * 0.7
        annual_ri_cost = ri_1year_rate * current_usage_hours
        
        savings = annual_on_demand_cost - annual_ri_cost
        
        return {
            'instance_type': instance_type,
            'annual_on_demand_cost': annual_on_demand_cost,
            'annual_ri_cost': annual_ri_cost,
            'potential_savings': savings,
            'payback_months': (current_usage_hours * on_demand_rate * 0.3) / (savings / 12)
        }
    
    def purchase_reserved_instance(self, instance_type: str, count: int,
                                   term_years: int, hourly_rate: float) -> bool:
        """Purchase reserved instance"""
        
        ri = ReservedInstance(instance_type, count, term_years, hourly_rate)
        self.reserved_instances.append(ri)
        
        return True
    
    def launch_spot_instances(self, instance_type: str, count: int, 
                             max_price: float) -> bool:
        """Launch spot instances"""
        
        spot = SpotInstance(instance_type, count, max_price)
        spot.is_active = True
        
        self.spot_instances.append(spot)
        
        return True
    
    def get_cost_optimization_summary(self) -> Dict:
        """Get optimization summary"""
        
        ri_total_cost = sum(ri.get_total_cost() for ri in self.reserved_instances)
        
        spot_hourly_cost = sum(s.current_price * s.count for s in self.spot_instances if s.is_active)
        spot_annual_cost = spot_hourly_cost * 365 * 24
        
        return {
            'reserved_instances_cost': ri_total_cost,
            'spot_instances_annual_cost': spot_annual_cost,
            'total_committed': ri_total_cost + spot_annual_cost,
            'num_reserved_instances': len(self.reserved_instances),
            'num_spot_instances': len([s for s in self.spot_instances if s.is_active])
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Cost Anomaly Detection & Forecasting

### Cost Analysis
```python
class CostAnomaly:
    """Cost anomaly detection"""
    
    def __init__(self, baseline_threshold: float = 1.5):
        self.baseline_costs: Dict[str, float] = {}
        self.daily_costs: List[Dict] = []
        self.threshold = baseline_threshold
    
    def set_baseline(self, resource_type: str, average_daily_cost: float):
        """Set baseline cost"""
        
        self.baseline_costs[resource_type] = average_daily_cost
    
    def detect_anomaly(self, resource_type: str, current_cost: float) -> bool:
        """Detect cost anomaly"""
        
        if resource_type not in self.baseline_costs:
            return False
        
        baseline = self.baseline_costs[resource_type]
        
        # Flag if cost exceeds threshold * baseline
        is_anomaly = current_cost > (baseline * self.threshold)
        
        self.daily_costs.append({
            'date': datetime.utcnow().isoformat(),
            'resource_type': resource_type,
            'cost': current_cost,
            'is_anomaly': is_anomaly
        })
        
        return is_anomaly
    
    def get_anomalies(self) -> List[Dict]:
        """Get detected anomalies"""
        
        return [c for c in self.daily_costs if c['is_anomaly']]

class CostForecaster:
    """Forecast future costs"""
    
    def __init__(self):
        self.historical_costs: List[Dict] = []
    
    def add_historical_cost(self, date: datetime, cost: float):
        """Add historical cost data"""
        
        self.historical_costs.append({
            'date': date,
            'cost': cost
        })
    
    def forecast_monthly_cost(self, trend_months: int = 3) -> Dict:
        """Forecast monthly cost"""
        
        if len(self.historical_costs) < trend_months:
            return {}
        
        # Get recent costs
        recent_costs = self.historical_costs[-trend_months:]
        costs = [c['cost'] for c in recent_costs]
        
        # Calculate trend
        average = sum(costs) / len(costs)
        trend = (costs[-1] - costs[0]) / trend_months if len(costs) > 1 else 0
        
        # Project forward 3 months
        forecast = {}
        for i in range(1, 4):
            forecast[f'month_{i}'] = average + (trend * i)
        
        return forecast
    
    def get_budget_alert(self, monthly_budget: float) -> Optional[str]:
        """Get budget alert"""
        
        if not self.historical_costs:
            return None
        
        current_month_cost = self.historical_costs[-1]['cost']
        
        if current_month_cost > monthly_budget * 0.8:
            overage = current_month_cost - monthly_budget
            return f"Alert: Budget exceeded by ${overage:.2f}"
        
        return None
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. FinOps Checklist

✅ **Cost Tracking**
- [ ] Tags implemented
- [ ] Cost allocation
- [ ] Tagging policy
- [ ] Cost centers defined
- [ ] Projects tracked

✅ **Optimization**
- [ ] Reserved instances analyzed
- [ ] Spot instances deployed
- [ ] Right-sizing done
- [ ] Savings calculated
- [ ] ROI verified

✅ **Chargeback**
- [ ] Model defined
- [ ] Departments set up
- [ ] Billing automated
- [ ] Reports generated
- [ ] Transparency achieved

✅ **Monitoring**
- [ ] Anomalies detected
- [ ] Forecasting active
- [ ] Budgets defined
- [ ] Alerts configured
- [ ] Dashboards built

✅ **Operational**
- [ ] Documentation
- [ ] Team training
- [ ] Governance rules
- [ ] Review process
- [ ] Continuous optimization
</VSCode.Cell>
```